# 01 — Run Experiment

Runs LLM API calls and saves results to Google Drive.

**First time?** See README.md for setup instructions.

## 1. Clone Repo & Mount Drive

In [2]:
import os

# Reset to a safe directory first
os.chdir('/content')

# Clone repo (fresh each time)
!rm -rf /content/authority-bias-llm
!git clone https://github.com/auertobias/authority-bias-llm.git /content/authority-bias-llm

# Change into the repo directory
os.chdir('/content/authority-bias-llm')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q google-genai openai anthropic

Cloning into '/content/authority-bias-llm'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 62 (delta 30), reused 26 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 165.88 KiB | 2.44 MiB/s, done.
Resolving deltas: 100% (30/30), done.
Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 8.2 MB/s eta 0:00:00


## 2. Configure

In [3]:
import os, sys
sys.path.insert(0, '.')

from src.config import DATA_PATH, RESULTS_PATH, N_REPS, PAUSE_SECONDS, CHECKPOINT_EVERY

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)
print(f"Data saves to: {DATA_PATH}")

Data saves to: /content/drive/MyDrive/PhD/2PhD 1Paper/data/


In [4]:
# ── API Keys ───────────────────────────────────────────────
# Use Colab Secrets (🔑 icon in sidebar) — add your keys there.
from google.colab import userdata

# Uncomment the keys you have:
GEMINI_API_KEY    = userdata.get('GEMINI_API_KEY')
DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
# OPENAI_API_KEY    = userdata.get('OPENAI_API_KEY')
# ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

## 3. Load Data & Build Trials

In [5]:
import pandas as pd

arguments   = pd.read_csv(DATA_PATH + "arguments.csv")
authorities = pd.read_csv(DATA_PATH + "authority.csv")

print(f"Arguments:   {len(arguments)} rows, {arguments['claim_id'].nunique()} claims")
print(f"Authorities: {len(authorities)} rows")

Arguments:   148 rows, 74 claims
Authorities: 12 rows


In [6]:
trials = []
trial_id = 0

for _, arg in arguments.iterrows():
    for _, auth in authorities.iterrows():
        if auth['branch'] == 'none':
            expertise = 'irrelevant'
        elif auth['branch'] == arg['branch']:
            expertise = 'relevant'
        else:
            expertise = 'irrelevant'

        trial_id += 1
        trials.append({
            'trial_id': trial_id,
            'argument_id': arg['id'],
            'claim': arg['claim'],
            'argument': arg['argument'],
            'stance': arg['stance'],
            'arg_type': arg['type'],
            'arg_branch': arg['branch'],
            'authority_id': auth['authority_id'],
            'authority_label': auth['label'].strip(),
            'status': auth['status'],
            'auth_branch': auth['branch'],
            'expertise': expertise,
        })

trials_df = pd.DataFrame(trials)
trials_df = trials_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total trials: {len(trials_df)}")
print(f"\nTrials per cell:")
print(trials_df.groupby(['status', 'expertise', 'arg_type']).size().unstack(fill_value=0))

Total trials: 1776

Trials per cell:
arg_type              descriptive  normative
status    expertise                         
high      irrelevant          222        222
          relevant             74         74
low       irrelevant          222        222
          relevant             74         74
medium    irrelevant           74         74
          relevant             74         74
very high irrelevant           74         74
          relevant             74         74


## 4. Choose Models & Run

In [7]:
import importlib
import src.models
importlib.reload(src.models)

<module 'src.models' from '/content/authority-bias-llm/src/models.py'>

In [8]:
from src.models import make_gemini_fn, make_gpt_fn, make_claude_fn, make_deepseek_fn
from src.prompts import build_prompt
from src.parsing import extract_rating, extract_weaknesses

# Uncomment the models you want to run:
MODELS = {
     'gemini': make_gemini_fn(GEMINI_API_KEY),
    # 'gpt':    make_gpt_fn(OPENAI_API_KEY),
    # 'claude': make_claude_fn(ANTHROPIC_API_KEY),
    # 'deepseek': make_deepseek_fn(DEEPSEEK_API_KEY),
}

if not MODELS:
    print("⚠️  No models selected! Uncomment at least one line above.")

In [9]:
import time
from datetime import datetime

for model_name, model_fn in MODELS.items():
    print(f"\n{'='*60}")
    print(f"  RUNNING: {model_name.upper()}")
    print(f"{'='*60}")

    results = []

    for rep in range(N_REPS):
        run_order = trials_df.sample(frac=1, random_state=rep*1000).reset_index(drop=True)
        print(f"\n--- Rep {rep+1}/{N_REPS} ({len(run_order)} trials) ---")

        for idx, row in run_order.iterrows():
            prompt = build_prompt(row)
            raw = model_fn(prompt)
            rating = extract_rating(raw)
            weaknesses = extract_weaknesses(raw)

            results.append({
                'trial_id': row['trial_id'],
                'rep': rep + 1,
                'model': model_name,
                'status': row['status'],
                'expertise': row['expertise'],
                'arg_type': row['arg_type'],
                'arg_branch': row['arg_branch'],
                'authority_label': row['authority_label'],
                'stance': row['stance'],
                'claim': row['claim'],
                'argument': row['argument'],
                'rating': rating,
                'weaknesses': weaknesses,
                'raw_response': raw,
            })

            n_done = idx + 1
            if n_done % 50 == 0:
                print(f"  {n_done}/{len(run_order)} done | last rating: {rating}")

            if n_done % CHECKPOINT_EVERY == 0:
                pd.DataFrame(results).to_csv(
                    DATA_PATH + f"checkpoint_{model_name}.csv", index=False
                )

            time.sleep(PAUSE_SECONDS)

    # Save final
    date_str = datetime.now().strftime("%Y%m%d")
    filename = f"data_{model_name}_{date_str}.csv"
    filepath = DATA_PATH + filename

    pd.DataFrame(results).to_csv(filepath, index=False)
    valid = sum(1 for r in results if r['rating'] is not None)
    print(f"\n✓ Saved: {filename}")
    print(f"  {len(results)} rows, {valid} valid ratings ({100*valid/len(results):.0f}%)")


  RUNNING: GEMINI

--- Rep 1/1 (1776 trials) ---
  50/1776 done | last rating: 35
  100/1776 done | last rating: 65
  150/1776 done | last rating: None
  200/1776 done | last rating: 55
  250/1776 done | last rating: 55
  300/1776 done | last rating: 70
  350/1776 done | last rating: 30
  400/1776 done | last rating: 35
  450/1776 done | last rating: 65
  500/1776 done | last rating: None
  550/1776 done | last rating: 60
  600/1776 done | last rating: 40
  650/1776 done | last rating: 40
  Gemini error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
  Gemini error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
  700/1776 done | last rating: 30
  750/1776 done | last rating: 35
  800/1776 done

## 5. Verify

In [10]:
import glob

data_files = sorted(glob.glob(DATA_PATH + "data_*.csv"))
print("Data files on Drive:")
for f in data_files:
    df_check = pd.read_csv(f)
    valid = df_check['rating'].notna().sum()
    print(f"  {os.path.basename(f):40s} → {len(df_check)} rows, {valid} valid")

Data files on Drive:
  data_deepseek_20260312.csv               → 1776 rows, 1776 valid
  data_gemini_20260313.csv                 → 1776 rows, 1483 valid
  data_gpt_20260311.csv                    → 1776 rows, 1776 valid
